In [5]:
!pip install sentence-transformers==4.1.0 

  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.6.0
    Uninstalling sentence-transformers-5.6.0:
      Successfully uninstalled sentence-transformers-5.6.0


In [8]:
import math

import numpy as np
import scipy
import torch
from sentence_transformers import SentenceTransformer



documents = [
    'Bugs introduced by the intern had to be squashed by the lead developer.',
    'Bugs found by the quality assurance engineer were difficult to debug.',
    'Bugs are common throughout the warm summer months, according to the entomologist.',
    'Bugs, in particular spiders, are extensively studied by arachnologists.'
]

model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

embeddings = model.encode(documents)

embeddings.shape

embeddings

array([[-0.22804375, -0.24647729, -0.00319285, ...,  0.45528072,
         0.6341972 ,  0.5375051 ],
       [-0.357916  , -0.32084027,  0.15963264, ..., -0.07050666,
         0.9275025 ,  0.3437728 ],
       [ 0.2030292 , -0.26898623,  0.1628513 , ..., -0.19651008,
        -0.03379876,  0.5956155 ],
       [-0.04264304, -0.4572165 , -0.09526499, ..., -0.58030725,
         0.17248413,  0.09127871]], dtype=float32)

In [10]:
#L2 (Euclidean) Distance Manual
def euclidean_distance_fn(vector1, vector2):
    squared_sum = sum((x - y) ** 2 for x, y in zip(vector1, vector2))
    return math.sqrt(squared_sum)


euclidean_distance_fn(embeddings[0], embeddings[1])

euclidean_distance_fn(embeddings[1], embeddings[0])

5.961788969115096

In [12]:
l2_dist_manual = np.zeros([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        l2_dist_manual[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])

l2_dist_manual

array([[0.        , 5.96178897, 7.33939916, 7.15578156],
       [5.96178897, 0.        , 7.76861591, 7.39359113],
       [7.33939916, 7.76861591, 0.        , 5.9199277 ],
       [7.15578156, 7.39359113, 5.9199277 , 0.        ]])

In [13]:
l2_dist_manual[0,1]

5.961788969115096

In [14]:
l2_dist_manual_improved = np.zeros([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        if j > i: # Calculate the upper triangle only
            l2_dist_manual_improved[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])
        elif i > j: # Copy the uper triangle to the lower triangle
            l2_dist_manual_improved[i,j] = l2_dist_manual[j,i]

l2_dist_manual_improved

array([[0.        , 5.96178897, 7.33939916, 7.15578156],
       [5.96178897, 0.        , 7.76861591, 7.39359113],
       [7.33939916, 7.76861591, 0.        , 5.9199277 ],
       [7.15578156, 7.39359113, 5.9199277 , 0.        ]])

In [15]:
#Usar una función de scipy para calcular la L2
l2_dist_scipy = scipy.spatial.distance.cdist(embeddings, embeddings, 'euclidean')
l2_dist_scipy

array([[0.        , 5.96178899, 7.33939915, 7.15578156],
       [5.96178899, 0.        , 7.76861588, 7.39359115],
       [7.33939915, 7.76861588, 0.        , 5.91992767],
       [7.15578156, 7.39359115, 5.91992767, 0.        ]])

In [16]:
np.allclose(l2_dist_manual, l2_dist_scipy)

True

In [17]:
def dot_product_fn(vector1, vector2):
    return sum(x * y for x, y in zip(vector1, vector2))

dot_product_fn(embeddings[0], embeddings[1])

18.535403203561145

In [18]:
dot_product_manual = np.empty([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        dot_product_manual[i,j] = dot_product_fn(embeddings[i], embeddings[j])

dot_product_manual

array([[33.74440412, 18.5354032 ,  8.56981599,  7.83093271],
       [18.5354032 , 38.86933037,  7.88997285,  8.66340567],
       [ 8.56981599,  7.88997285, 37.26200782, 17.6695677 ],
       [ 7.83093271,  8.66340567, 17.6695677 , 33.12267103]])

In [19]:
# Matrix multiplication operator
dot_product_operator = embeddings @ embeddings.T
dot_product_operator

array([[33.744396 , 18.535402 ,  8.569816 ,  7.830931 ],
       [18.535402 , 38.869324 ,  7.8899693,  8.663405 ],
       [ 8.569816 ,  7.8899693, 37.262016 , 17.669569 ],
       [ 7.830931 ,  8.663405 , 17.669569 , 33.122665 ]], dtype=float32)

In [20]:
np.allclose(dot_product_manual, dot_product_operator, atol=1e-05)

True

In [21]:
# Equivalent to `np.matmul()` if both arrays are 2-D:
np.matmul(embeddings,embeddings.T)

array([[33.744396 , 18.535402 ,  8.569816 ,  7.830931 ],
       [18.535402 , 38.869324 ,  7.8899693,  8.663405 ],
       [ 8.569816 ,  7.8899693, 37.262016 , 17.669569 ],
       [ 7.830931 ,  8.663405 , 17.669569 , 33.122665 ]], dtype=float32)

In [22]:
# `np.dot` returns an identical result, but `np.matmul` is recommended if both arrays are 2-D:
np.dot(embeddings,embeddings.T)

array([[33.744396 , 18.535402 ,  8.569816 ,  7.830931 ],
       [18.535402 , 38.869324 ,  7.8899693,  8.663405 ],
       [ 8.569816 ,  7.8899693, 37.262016 , 17.669569 ],
       [ 7.830931 ,  8.663405 , 17.669569 , 33.122665 ]], dtype=float32)

In [24]:
#The dot product between two vectors provides a similarity score. 
#If, on the other hand, we would like a distance, we can simply take the negative of the dot product:

dot_product_distance = -dot_product_manual
dot_product_distance

array([[-33.74440412, -18.5354032 ,  -8.56981599,  -7.83093271],
       [-18.5354032 , -38.86933037,  -7.88997285,  -8.66340567],
       [ -8.56981599,  -7.88997285, -37.26200782, -17.6695677 ],
       [ -7.83093271,  -8.66340567, -17.6695677 , -33.12267103]])

In [28]:
# L2 norms
l2_norms = np.sqrt(np.sum(embeddings**2, axis=1))
l2_norms



array([5.808993 , 6.2345276, 6.1042614, 5.75523  ], dtype=float32)

In [27]:

# L2 norms reshaped
l2_norms_reshaped = l2_norms.reshape(-1,1)
l2_norms_reshaped

array([[5.808993 ],
       [6.2345276],
       [6.1042614],
       [5.75523  ]], dtype=float32)

In [29]:
normalized_embeddings_manual = embeddings/l2_norms_reshaped
normalized_embeddings_manual

array([[-0.03925702, -0.0424303 , -0.00054964, ...,  0.07837515,
         0.10917506,  0.09252983],
       [-0.05740868, -0.05146184,  0.02560461, ..., -0.01130906,
         0.1487687 ,  0.05514015],
       [ 0.03326024, -0.04406532,  0.0266783 , ..., -0.03219228,
        -0.00553691,  0.09757372],
       [-0.00740944, -0.07944366, -0.01655277, ..., -0.10083129,
         0.02996998,  0.01586013]], dtype=float32)

In [30]:
np.sqrt(np.sum(normalized_embeddings_manual**2, axis=1))

array([1.        , 0.99999994, 1.        , 1.        ], dtype=float32)

In [31]:
normalized_embeddings_torch = torch.nn.functional.normalize(
    torch.from_numpy(embeddings)
).numpy()
normalized_embeddings_torch

array([[-0.03925702, -0.04243029, -0.00054964, ...,  0.07837515,
         0.10917506,  0.09252982],
       [-0.05740868, -0.05146184,  0.02560461, ..., -0.01130906,
         0.1487687 ,  0.05514015],
       [ 0.03326024, -0.04406532,  0.0266783 , ..., -0.03219228,
        -0.00553691,  0.09757372],
       [-0.00740944, -0.07944366, -0.01655277, ..., -0.10083129,
         0.02996998,  0.01586013]], dtype=float32)

In [32]:
np.allclose(normalized_embeddings_manual, normalized_embeddings_torch)

True

In [33]:
dot_product_fn(normalized_embeddings_manual[0], normalized_embeddings_manual[1])

0.5117968921025522

In [34]:
cosine_similarity_manual = np.empty([4,4])
for i in range(normalized_embeddings_manual.shape[0]):
    for j in range(normalized_embeddings_manual.shape[0]):
        cosine_similarity_manual[i,j] = dot_product_fn(
            normalized_embeddings_manual[i], 
            normalized_embeddings_manual[j]
        )

cosine_similarity_manual

array([[1.00000018, 0.51179689, 0.24167823, 0.23423402],
       [0.51179689, 0.99999991, 0.20731887, 0.24144734],
       [0.24167823, 0.20731887, 1.00000002, 0.50295615],
       [0.23423402, 0.24144734, 0.50295615, 0.99999998]])

In [35]:
cosine_similarity_operator = normalized_embeddings_manual @ normalized_embeddings_manual.T
cosine_similarity_operator

array([[1.0000002 , 0.51179695, 0.24167828, 0.23423404],
       [0.51179695, 1.0000001 , 0.20731883, 0.24144733],
       [0.24167828, 0.20731883, 1.0000006 , 0.50295603],
       [0.23423404, 0.24144733, 0.50295603, 0.99999976]], dtype=float32)

In [36]:
np.allclose(cosine_similarity_manual, cosine_similarity_operator)

True

In [37]:
1 - cosine_similarity_manual

array([[-1.82379040e-07,  4.88203108e-01,  7.58321765e-01,
         7.65765979e-01],
       [ 4.88203108e-01,  8.88251293e-08,  7.92681131e-01,
         7.58552659e-01],
       [ 7.58321765e-01,  7.92681131e-01, -2.06883648e-08,
         4.97043852e-01],
       [ 7.65765979e-01,  7.58552659e-01,  4.97043852e-01,
         1.59195790e-08]])

In [38]:
query_embedding = model.encode(
    ["Who is responsible for a coding project and fixing others' mistakes?"]
)

# Second, normalize the query embedding:
normalized_query_embedding = torch.nn.functional.normalize(
    torch.from_numpy(query_embedding)
).numpy()

# Third, calculate the cosine similarity between the documents and the query by using the dot product:
cosine_similarity_q3 = normalized_embeddings_manual @ normalized_query_embedding.T

# Fourth, find the position of the vector with the highest cosine similarity:
highest_cossim_position = cosine_similarity_q3.argmax()

# Fifth, find the document in that position in the `documents` array:
documents[highest_cossim_position]

# As you can see, the query retrieved the document `Bugs introduced by the intern had to be squashed by the lead developer.` which is what we would expect.

'Bugs introduced by the intern had to be squashed by the lead developer.'